# Financial Document Analysis Agent

**CSC3173 Artificial Intelligence: Programming Assignment 02**

This notebook builds an agent that answers financial-analysis questions about **Apple Inc.**
using its annual reports (Form 10-K) for fiscal years **2023, 2024 and 2025**.

The design follows the Medical Document (WHO statistics) Agent from class: PDFs are split into
chunks, embedded and indexed with FAISS, and a LangGraph loop lets a planner choose between
`RETRIEVE`, `CALCULATE`, `REPORT` and `FINISH`. The health-specific parts were replaced with
financial ones: a table-aware retrieval tool for the 10-K net sales tables, a Python calculation
tool, and a report generator that writes a PDF with charts and page citations.

Notebook outline:

| Stage | Section |
|---|---|
| 0 | Agentic components (Medical agent vs Financial agent) |
| A | Document processing |
| B | Semantic retrieval |
| C | Financial retrieval tool |
| D | Financial calculation tool |
| E | Report generation tool |
| F | LangGraph agent (state, planner, nodes, routing) |
| G | Test questions Q1 to Q3 and PDF report |

## Setup
Mount Google Drive (on Colab) and set the project paths. The three 10-K PDFs go in `Finance_AI_Agent/data/` (or directly in `Finance_AI_Agent/`).

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Demo/AI/Finance_AI_Agent"
else:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = PROJECT_DIR
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Data dir   :", DATA_DIR)
print("Output dir :", OUTPUT_DIR)

### Package installation

In [ ]:
%pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate
%pip install -q langgraph
%pip install -q fpdf2

In [ ]:
import fitz  # PyMuPDF
import re, glob, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

## A. Document Processing

Each PDF is opened with PyMuPDF and the text of every page is kept together with its metadata
(source file, page number, fiscal year).

The class notebook took the year from the file name. Here the fiscal year is read from the cover
page of the filing (*"For the fiscal year ended September 27, 2025"*), so renamed files still get
the correct year. Only PDFs whose first page says *FORM 10-K* are loaded, which keeps the
generated report PDF out of the index if it ends up in the same folder.

In [ ]:
def is_10k_filing(doc) -> bool:
    return "FORM 10-K" in doc[0].get_text().upper()

def detect_fiscal_year(doc, path: str) -> str:
    cover = doc[0].get_text().replace("\xa0", " ")
    match = re.search(r"fiscal year ended\s+[A-Za-z]+\s+\d{1,2},\s+(20\d{2})", cover, re.IGNORECASE)
    if match:
        return match.group(1)
    match = re.search(r"(20\d{2})", os.path.basename(path))
    return match.group(1) if match else "unknown"

def clean_page_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)
    return text.strip()

def extract_pdf_pages(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    if not is_10k_filing(doc):
        return []
    fiscal_year = detect_fiscal_year(doc, pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        text = clean_page_text(page.get_text())
        if text:
            pages.append({"text": text, "page": i, "year": fiscal_year,
                          "source": os.path.basename(pdf_path)})
    return pages

pdf_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.pdf")))
assert pdf_paths, f"No PDF files found in {DATA_DIR}"

all_pages = []
for path in pdf_paths:
    pages = extract_pdf_pages(path)
    if pages:
        all_pages.extend(pages)
        print(f"{os.path.basename(path):28s} -> FY{pages[0]['year']}, {len(pages)} pages")

AVAILABLE_YEARS = sorted({p["year"] for p in all_pages if p["year"] != "unknown"})
print("Fiscal years available:", AVAILABLE_YEARS)

In [ ]:
# quick lookup of the full text of any page, used later by the table parser
page_lookup = {(p["source"], p["page"]): p for p in all_pages}

summary = (pd.DataFrame(all_pages)
           .assign(words=lambda d: d["text"].str.split().str.len())
           .groupby(["year", "source"])
           .agg(pages=("page", "count"), words=("words", "sum")))
summary

### Chunking with source / year / page metadata

Pages are split into overlapping word windows. Every chunk keeps the metadata of its page so
answers can always be traced back to a page in a filing.

In [ ]:
def chunk_text(text: str, chunk_size: int = 250, overlap: int = 50) -> list[str]:
    words = text.split()
    if len(words) <= chunk_size:
        return [" ".join(words)]
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
        start += chunk_size - overlap
    return chunks

def build_chunks(pages: list[dict]) -> list[dict]:
    out = []
    for page in pages:
        for piece in chunk_text(page["text"]):
            out.append({"text": piece, "page": page["page"], "year": page["year"],
                        "source": page["source"]})
    return out

chunks = build_chunks(all_pages)
print(f"Total chunks: {len(chunks)}")
print(pd.Series([c["year"] for c in chunks]).value_counts().sort_index().to_string())

## B. Semantic Retrieval

Chunks are embedded with `all-MiniLM-L6-v2` (normalised vectors), so the inner product in a
FAISS `IndexFlatIP` index is the cosine similarity. `semantic_search` can search the whole
collection or only the chunks of one fiscal year, which plays the same role as
`retrieve_within_year` in the class notebook.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

embeddings = embed_model.encode([c["text"] for c in chunks], batch_size=64, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

# chunk ids per fiscal year for year-restricted search
year_to_ids = {}
for i, c in enumerate(chunks):
    year_to_ids.setdefault(c["year"], []).append(i)

print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={embeddings.shape[1]}, device={device}")

In [ ]:
def semantic_search(query: str, k: int = 5, year: str | None = None) -> list[dict]:
    q = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    if year is None:
        scores, idxs = faiss_index.search(q, k)
        hits = list(zip(scores[0], idxs[0]))
    else:
        ids = year_to_ids.get(year, [])
        if not ids:
            return []
        scores = embeddings[ids] @ q[0]
        top = np.argsort(-scores)[:k]
        hits = [(scores[j], ids[j]) for j in top]
    return [{**chunks[i], "score": float(s)} for s, i in hits]

for hit in semantic_search("total net sales by product category", k=3, year="2025"):
    print(f"[{hit['source']} FY{hit['year']} p.{hit['page']}] score={hit['score']:.3f}")
    print("   ", hit["text"][:160], "...")

## Qwen loading

Qwen2.5-1.5B-Instruct is the language model of the agent (same as in class). It is used for
understanding the question, for the planner, as a fallback extractor and for writing the
analysis text. It is **not** used for any arithmetic.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto")
qwen_model.eval()
print("Qwen loaded on", qwen_model.device)

def generate_with_qwen(prompt: str, system_prompt: str = "You are a careful financial analyst.",
                       max_new_tokens: int = 300) -> str:
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}]
    text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(text, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        output = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return qwen_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## C. Financial Retrieval Tool

Financial questions need exact figures, and the 10-K keeps them in tables. In the class notebook,
values were pulled out of free text by the LLM plus a regex, which caused two problems in its output:
the year *2024* was read as the indicator value, and the 2023 value was wrong.

The financial retrieval tool works in two steps:

1. **Semantic search** (restricted to one fiscal year) finds the page with the required table.
2. **Table parser:** on that page the rows of the table are read in order and the numbers are
   matched to the column years from the table header (`2025 Change 2024 Change 2023`).
   Percent-change columns and footnote markers are removed first, so only dollar amounts
   (in millions) are kept.

Two tables are supported: *net sales by category* (iPhone, Mac, iPad, Wearables/Home/Accessories,
Services) and *net sales by reportable segment* (Americas, Europe, Greater China, Japan,
Rest of Asia Pacific). If the parser cannot read a row, Qwen is asked to extract the value from
the retrieved text, and the answer is only accepted if that number really appears in the text.

In [ ]:
TABLE_SPECS = {
    "category": {
        "title": "Net sales by category",
        "anchor": "net sales by category",
        "query": "net sales by category iPhone Mac iPad Wearables Home and Accessories Services total net sales",
        "rows": ["iPhone", "Mac", "iPad", "Wearables, Home and Accessories", "Services", "Total net sales"],
    },
    "segment": {
        "title": "Net sales by reportable segment",
        "anchor": "net sales by reportable segment",
        "query": "net sales by reportable segment Americas Europe Greater China Japan Rest of Asia Pacific",
        "rows": ["Americas", "Europe", "Greater China", "Japan", "Rest of Asia Pacific", "Total net sales"],
    },
}

AMOUNT = r"\d{1,3}(?:,\d{3})+"   # amounts in the tables are in millions, e.g. 209,586

def parse_financial_table(page_text: str, spec: dict) -> dict | None:
    text = re.sub(r"\s+", " ", page_text)
    start = text.lower().find(spec["anchor"])
    if start < 0:
        return None
    body = text[start:]
    header = re.search(r"(20\d{2})\s+Change\s+(20\d{2})\s+Change\s+(20\d{2})", body)
    if not header:
        return None
    col_years = list(header.groups())
    body = body[header.end():]
    body = re.sub(r"\(?\d+(?:\.\d+)?\)?\s*%|—\s*%|-\s*%", " ", body)   # drop percent-change columns
    body = re.sub(r"\(\d\)", " ", body)                                      # drop footnote markers

    rows, pos = {}, 0
    for label in spec["rows"]:
        at = body.find(label, pos)
        if at < 0:
            continue
        after = body[at + len(label): at + len(label) + 90]
        amounts = re.findall(AMOUNT, after)[:len(col_years)]
        if len(amounts) == len(col_years):
            rows[label] = {y: float(a.replace(",", "")) for y, a in zip(col_years, amounts)}
            pos = at + len(label)
    return {"columns": col_years, "rows": rows} if rows else None

In [ ]:
def llm_extract_value(label: str, context: str, year: str) -> float | None:
    # fallback extractor: Qwen reads the text, Python checks the number really is in the text
    prompt = f'''From the text below, give the value of "{label}" for fiscal year {year},
in millions of US dollars. Reply in exactly this format:
Value: <number>
If the value is not in the text, reply: Value: not found

Text:
{context[:3000]}
'''
    answer = generate_with_qwen(prompt, max_new_tokens=20)
    match = re.search(r"Value:\s*\$?\s*([\d,]+(?:\.\d+)?)", answer)
    if not match:
        return None
    raw = match.group(1)
    value = float(raw.replace(",", ""))
    if 1990 <= value <= 2100 and "," not in raw:   # a year, not an amount
        return None
    if raw not in context and f"{value:,.0f}" not in context:
        return None
    return value

In [ ]:
def retrieve_financial_table(table: str, year: str, attempt: int = 1) -> dict | None:
    # Attempt 1: search inside the filing for that fiscal year.
    # Attempt 2: search every filing; later 10-Ks repeat earlier years as comparative columns.
    spec = TABLE_SPECS[table]
    hits = (semantic_search(spec["query"], k=8, year=year) if attempt == 1
            else semantic_search(spec["query"], k=20))
    seen = set()
    for hit in hits:
        key = (hit["source"], hit["page"])
        if key in seen:
            continue
        seen.add(key)
        parsed = parse_financial_table(page_lookup[key]["text"], spec)
        if not parsed or year not in parsed["columns"]:
            continue
        values = {label: cols[year] for label, cols in parsed["rows"].items()}
        missing = [r for r in spec["rows"] if r not in values]
        for label in missing:
            value = llm_extract_value(label, page_lookup[key]["text"], year)
            if value is not None:
                values[label] = value
        return {"table": table, "year": year, "values": values, "source": hit["source"],
                "page": hit["page"], "filing_year": hit["year"], "score": hit["score"],
                "method": "table parser" + (" + LLM fallback" if missing else ""),
                "all_columns": parsed["rows"]}
    return None

def financial_retrieval_tool(query: str, k: int = 4, year: str | None = None) -> str:
    # text retrieval with citations, used as context for the written analysis
    return "\n\n".join(f"[{h['source']} (FY{h['year']}) p.{h['page']}]\n{h['text']}"
                       for h in semantic_search(query, k=k, year=year))

In [ ]:
# test: category and segment tables for every fiscal year
for table in TABLE_SPECS:
    for year in AVAILABLE_YEARS:
        r = retrieve_financial_table(table, year)
        print(f"{table:8s} FY{year}: {r['source']} p.{r['page']} ({r['method']})")
        print("   ", {k: f"{v:,.0f}" for k, v in r["values"].items()})

In [ ]:
# consistency check: the same fiscal year appears in up to three filings (as a comparative column)
check_rows = []
for year in AVAILABLE_YEARS:
    for filing in AVAILABLE_YEARS:
        r = retrieve_financial_table("category", filing)
        if r and year in r["all_columns"]["Total net sales"]:
            check_rows.append({"fiscal_year": year, "reported_in": f"FY{filing} 10-K",
                               "total_net_sales_$m": r["all_columns"]["Total net sales"][year]})
pd.DataFrame(check_rows).pivot(index="fiscal_year", columns="reported_in", values="total_net_sales_$m")

In [ ]:
# test of the LLM fallback extractor on a retrieved page (only used when the parser misses a row)
sample = retrieve_financial_table("category", AVAILABLE_YEARS[-1])
sample_text = page_lookup[(sample["source"], sample["page"])]["text"]
llm_value = llm_extract_value("iPhone net sales", sample_text, AVAILABLE_YEARS[-1])
print("Qwen fallback, iPhone:", llm_value)
print("Table parser,  iPhone:", sample["values"]["iPhone"])
print("Agree:", llm_value == sample["values"]["iPhone"])

## D. Financial Calculation Tool

All arithmetic is done in Python. The LLM only receives the finished numbers.

* percentage change: `((V_new - V_old) / V_old) x 100`
* absolute change
* compound annual growth rate (CAGR)
* contribution share of each category or region to total net sales
* growth of every line item between two years, with the highest and lowest performer

In [ ]:
def percent_change(v_old: float, v_new: float) -> float | None:
    return None if not v_old else (v_new - v_old) / v_old * 100

def absolute_change(v_old: float, v_new: float) -> float:
    return v_new - v_old

def cagr(v_old: float, v_new: float, n_years: int) -> float | None:
    if not v_old or n_years <= 0:
        return None
    return ((v_new / v_old) ** (1 / n_years) - 1) * 100

def contribution_shares(values: dict, total: float) -> dict:
    return {k: v / total * 100 for k, v in values.items()}

def growth_table(old: dict, new: dict) -> dict:
    return {k: {"old": old[k], "new": new[k], "abs_change": absolute_change(old[k], new[k]),
                "pct_change": percent_change(old[k], new[k])}
            for k in old if k in new}

def financial_calculator(operation: str, **kw) -> dict:
    if operation == "total_change":
        series, y_old, y_new = kw["series"], kw["year_old"], kw["year_new"]
        v_old, v_new = series[y_old], series[y_new]
        yoy = {f"{a}->{b}": percent_change(series[a], series[b])
               for a, b in zip(sorted(series), sorted(series)[1:])}
        return {"operation": operation, "year_old": y_old, "year_new": y_new,
                "value_old": v_old, "value_new": v_new,
                "absolute_change": absolute_change(v_old, v_new),
                "percent_change": percent_change(v_old, v_new),
                "cagr": cagr(v_old, v_new, int(y_new) - int(y_old)),
                "year_over_year": yoy, "series": series}

    if operation == "top_contributor":
        items, total = kw["items"], kw["total"]
        shares = contribution_shares(items, total)
        ranking = sorted(items, key=items.get, reverse=True)
        return {"operation": operation, "year": kw["year"], "total": total,
                "items": items, "shares": shares, "ranking": ranking,
                "top": ranking[0], "top_value": items[ranking[0]], "top_share": shares[ranking[0]],
                "sum_check": sum(items.values()) - total}

    if operation == "growth_ranking":
        table = growth_table(kw["old"], kw["new"])
        ranking = sorted(table, key=lambda k: table[k]["pct_change"], reverse=True)
        share_old = contribution_shares(kw["old"], kw["total_old"])
        share_new = contribution_shares(kw["new"], kw["total_new"])
        return {"operation": operation, "year_old": kw["year_old"], "year_new": kw["year_new"],
                "growth": table, "ranking": ranking, "fastest": ranking[0], "slowest": ranking[-1],
                "share_old": share_old, "share_new": share_new,
                "total_pct_change": percent_change(kw["total_old"], kw["total_new"])}

    raise ValueError(f"Unknown operation: {operation}")

In [ ]:
# sanity checks on the calculator
assert percent_change(100, 110) == 10
assert absolute_change(383_285, 416_161) == 32_876
assert round(cagr(100, 121, 2), 6) == 10
assert percent_change(0, 5) is None
demo = financial_calculator("top_contributor", year="demo", items={"A": 30, "B": 70}, total=100)
assert demo["top"] == "B" and demo["top_share"] == 70
print("Calculator checks passed.")